In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("All good:", os.getcwd())

All good: /Users/Ferryadmin/Desktop/Data Center-Renewables Exploration/notebooks


In [2]:
import pandas as pd
#read master table in
master = pd.read_csv("../data/processed/data_center_energy_master_2025.csv")


In [3]:
water = pd.read_csv('../data/raw/published_annual_thermoelectric_water_use_estimates_2008-2020.csv')
print(water.shape)
print(water.columns.to_list)
water.head()


(14371, 18)
<bound method IndexOpsMixin.tolist of Index(['Plant.Code', 'huc_12', 'YEAR', 'Plant.Name', 'County', 'State',
       'Name.of.Water.Source', 'coolingType', 'ModelType',
       'Plant.level_dom_fuel', 'general_mover', 'Net.Generation.Year.To.Date',
       'cu_mgd', 'cu_lower_mgd', 'cu_upper_mgd', 'wd_mgd', 'wd_lower_mgd',
       'wd_upper_mgd'],
      dtype='str')>


,Plant.Code,huc_12,YEAR,Plant.Name,County,State,Name.of.Water.Source,coolingType,ModelType,Plant.level_dom_fuel,general_mover,Net.Generation.Year.To.Date,cu_mgd,cu_lower_mgd,cu_upper_mgd,wd_mgd,wd_lower_mgd,wd_upper_mgd
0,3,31602040106,2008,Barry,Mobile,AL,Mobile River,Complex,complex,multi-fuel,complex,14031145.0,11.576264,9.125151,14.027377,802.118534,367.368337,5877.849890
1,7,31501060204,2008,Gadsden,Etowah,AL,Coosa River,Once-through fresh,river,coal,ST,599689.0,0.820784,0.640212,1.001357,75.769756,34.554327,561.698436
2,8,31601090604,2008,Gorgas,Walker,AL,Warrior River,Once-through fresh,river,coal,ST,7798412.0,7.031521,5.484587,8.578456,656.859325,299.516331,4871.968301
3,10,31601130806,2008,Greene County,Greene,AL,Warrior River,Once-through fresh,river,coal,ST,3352933.0,2.966554,2.313912,3.619196,274.086846,125.035165,2029.659703
4,26,31501070304,2008,E C Gaston,Shelby,AL,Coosa River,Complex,complex,coal,ST,11411275.0,11.316589,9.092986,13.540192,441.004731,202.993483,3216.672166


In [4]:
#filter only the year 2020
water_2020 = water[water['YEAR'] == 2020]
print(water_2020.shape)

(958, 18)


In [5]:
#sum withdrawals by state
water_state = water_2020.groupby('State')['wd_mgd'].sum().reset_index()
water_state.columns = ['state_abbrev', 'total_wd_mgd']
print(water_state.shape)
print(water_state.head())

(48, 2)
  state_abbrev  total_wd_mgd
0           AL   3149.727582
1           AR   1088.739099
2           AZ    168.736235
3           CA   2347.040027
4           CO     50.923411


In [6]:
#match the state abbreviations in water use file to state names in master table 
state_mapping = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'FL': 'Florida', 'GA': 'Georgia', 'ID': 'Idaho', 'IL': 'Illinois',
    'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas', 'KY': 'Kentucky',
    'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland', 'MA': 'Massachusetts',
    'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire',
    'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina',
    'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma', 'OR': 'Oregon',
    'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
    'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia'
}

water_state['state_or_region'] = water_state['state_abbrev'].map(state_mapping)
print(water_state.head())

  state_abbrev  total_wd_mgd state_or_region
0           AL   3149.727582         Alabama
1           AR   1088.739099        Arkansas
2           AZ    168.736235         Arizona
3           CA   2347.040027      California
4           CO     50.923411        Colorado


In [7]:
#inspect the table to make sure no nulls, everything looks good
master.info()
master.head()
master.describe()
master.columns

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   state_or_region                 30 non-null     str    
 1   year                            30 non-null     int64  
 2   price_sector                    30 non-null     str    
 3   price                           30 non-null     float64
 4   renewable_sector                30 non-null     str    
 5   renewable_generation            30 non-null     float64
 6   total_generation_sector         30 non-null     str    
 7   total_generation                30 non-null     float64
 8   dc_sector                       30 non-null     str    
 9   data_center_facility_count      30 non-null     int64  
 10  data_center_estimated_power_mw  30 non-null     float64
 11  data_center_selected_power_mw   30 non-null     float64
 12  renewable_share                 30 non-null     f

Index(['state_or_region', 'year', 'price_sector', 'price', 'renewable_sector',
       'renewable_generation', 'total_generation_sector', 'total_generation',
       'dc_sector', 'data_center_facility_count',
       'data_center_estimated_power_mw', 'data_center_selected_power_mw',
       'renewable_share'],
      dtype='str')

In [8]:
master_states = set(master['state_or_region'].unique())
water_states = set(water_state['state_or_region'].unique())

print("In master but not water:", master_states - water_states)
print("In water but not master:", water_states - master_states)

In master but not water: set()
In water but not master: {'New Hampshire', 'West Virginia', 'Louisiana', 'Kentucky', 'South Dakota', 'New Mexico', 'Kansas', 'North Dakota', 'Rhode Island', 'Arkansas', 'Vermont', 'Mississippi', 'Montana', 'South Carolina', 'Maine', 'Alabama', 'Wyoming', 'Tennessee'}


In [9]:
df_master = pd.merge(master, water_state[['state_or_region', 'total_wd_mgd']], 
                     on='state_or_region', 
                     how='left')

print(df_master.shape)
print(df_master[['state_or_region', 'total_wd_mgd']].head(10))

(30, 14)
  state_or_region  total_wd_mgd
0         Arizona    168.736235
1      California   2347.040027
2        Colorado     50.923411
3     Connecticut   1943.954017
4        Delaware     21.890778
5         Florida   6065.749245
6         Georgia    149.753327
7           Idaho      2.027209
8        Illinois   4949.742135
9         Indiana   2940.347926


In [10]:
df_master.to_csv('../data/processed/master_df_final.csv', index=False)
print("Saved successfully")

Saved successfully


In [11]:
print(df_master['total_wd_mgd'].isnull().sum())

0


In [12]:
# general correlation matrix between numeric values: not applied to project focus, just practice
df_numeric= df_master.select_dtypes(include='number')
corr_matrix= df_numeric.corr()
print(corr_matrix)

                                year     price  renewable_generation  \
year                             NaN       NaN                   NaN   
price                            NaN  1.000000              0.648054   
renewable_generation             NaN  0.648054              1.000000   
total_generation                 NaN  0.725941              0.930082   
data_center_facility_count       NaN -0.138494              0.146932   
data_center_estimated_power_mw   NaN -0.202111              0.070629   
data_center_selected_power_mw    NaN -0.253041             -0.003011   
renewable_share                  NaN -0.193043              0.031887   
total_wd_mgd                     NaN  0.010612              0.224696   

                                total_generation  data_center_facility_count  \
year                                         NaN                         NaN   
price                                   0.725941                   -0.138494   
renewable_generation                   

In [13]:
#Data center demand + total generation, renewable share, and electricity price correlation
cols_1= ['data_center_selected_power_mw', 'renewable_share', 'price', 'total_generation']
df_matrix1= df_master[cols_1]
corr_1= df_matrix1.corr()
print(corr_1)


                               data_center_selected_power_mw  renewable_share  \
data_center_selected_power_mw                       1.000000        -0.174433   
renewable_share                                    -0.174433         1.000000   
price                                              -0.253041        -0.193043   
total_generation                                    0.059199        -0.180984   

                                  price  total_generation  
data_center_selected_power_mw -0.253041          0.059199  
renewable_share               -0.193043         -0.180984  
price                          1.000000          0.725941  
total_generation               0.725941          1.000000  


Finding 1 — data_center_selected_power_mw vs price: -0.25
A negative correlation was observed between data center selected power and electricity price. I would expect these to be positively correlated — higher demand typically drives up price — however this may reflect deliberate site selection by data center operators who seek out states with cheap, abundant power. In reality, growing data center demand is placing upward pressure on prices in many markets, which this small sample may not capture.
With only 30 observations this relationship is exploratory only and no realistic conclusions can be drawn.



Finding 2 — data_center_selected_power_mw vs renewable_share: -0.17
Data center selected power and renewable share show a weak negative correlation. Intuitively I would expect states with higher DC demand to invest more heavily in renewables to bridge the supply gap cheaply, but this sample suggests the opposite — states with higher sampled demand tend to have lower renewable share. This may reflect that data centers are currently siting in fossil-fuel dominated grids where power is cheapest.
With only 30 observations this relationship is exploratory only and no realistic conclusions can be drawn.



Finding 3 — price vs total_generation: +0.73
The strongest relationship in this matrix. Rather than higher supply driving lower prices as basic economics might suggest, both variables appear to be driven by a confounding variable — state size. Larger states with greater economic activity and population have both higher total generation and higher electricity prices, as demand is placing strain on the grid that supply is struggling to meet.
With only 30 observations this relationship is exploratory only and no realistic conclusions can be drawn.

In [14]:
cols_2= ['total_wd_mgd', 'renewable_share', 'data_center_selected_power_mw', 'total_generation']
df_matrix2= df_master[cols_2]
corr_2= df_matrix2.corr()
print(corr_2)

                               total_wd_mgd  renewable_share  \
total_wd_mgd                       1.000000        -0.361748   
renewable_share                   -0.361748         1.000000   
data_center_selected_power_mw      0.358009        -0.174433   
total_generation                   0.375764        -0.180984   

                               data_center_selected_power_mw  total_generation  
total_wd_mgd                                        0.358009          0.375764  
renewable_share                                    -0.174433         -0.180984  
data_center_selected_power_mw                       1.000000          0.059199  
total_generation                                    0.059199          1.000000  


Finding 1 — total_wd_mgd vs renewable_share: -0.36

States with a higher renewable energy share tend to have lower thermoelectric water withdrawals. This is consistent with the known water intensity of thermal generation — coal, natural gas, and nuclear plants require significant cooling water, while solar and wind do not. This is one of the stronger relationships in the dataset (r = -0.36) and aligns with established energy-water nexus literature. With only 30 observations this relationship is exploratory only and no realistic conclusions can be drawn.

Finding 2 — total_wd_mgd vs data_center_selected_power_mw: +0.36

States with higher sampled data center demand show slightly higher thermoelectric water withdrawals. This likely reflects co-location in large industrial states with fossil-fuel heavy grids rather than a direct causal link. With a larger sample this relationship would be worth examining further. With only 30 observations this relationship is exploratory only and no realistic conclusions can be drawn.

Finding 3 — total_wd_mgd vs total_generation: +0.38

Total water withdrawal correlates positively with total generation, consistent with the expectation that larger electricity systems — which rely more heavily on thermal generation — withdraw more water for cooling. With only 30 observations this relationship is exploratory only and no realistic conclusions can be drawn.


In [ ]:
#quick stats summary
df.describe()


,year,price,renewable_generation,total_generation,data_center_facility_count,data_center_estimated_power_mw,data_center_selected_power_mw,renewable_share
count,30.0,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000
mean,2025.0,13.124278,138.800000,481.433333,6.366667,252.778667,341.147967,0.353167
std,0.0,4.540858,220.938188,613.535862,8.965387,387.461424,514.724201,0.290912
min,2025.0,8.639167,-1.000000,-3.000000,1.000000,0.600000,0.500000,0.000000
25%,2025.0,10.341250,9.500000,83.000000,1.000000,34.367500,34.000000,0.079464
50%,2025.0,11.910833,31.000000,236.500000,3.000000,98.445000,101.250000,0.358732
75%,2025.0,13.676458,141.500000,697.500000,6.500000,251.965000,377.735000,0.484523
max,2025.0,26.192500,858.000000,2276.000000,40.000000,1812.730000,2078.910000,1.000000
